# Ladung Probability Threshold Analysis
Analyze how precision and recall change at different probability thresholds using the DVC pipeline test data and trained model.

In [1]:
import numpy as np
import scipy.sparse as sp
import dill
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Paths (from DVC pipeline: prepare.target = ladung)
MODEL_PATH = "../../../models/classification/ladung_rf_classifier.dill"
TEST_FEATURES_PATH = "../../../data/featurized/ladung/test_features.npz"
TEST_LABELS_PATH = "../../../data/featurized/ladung/test_labels.npy"

# Load model
with open(MODEL_PATH, "rb") as f:
    model = dill.load(f)
print(f"Model loaded: {type(model).__name__}")

# Load test data
X_test = sp.load_npz(TEST_FEATURES_PATH)
y_test = np.load(TEST_LABELS_PATH)
print(f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"Label distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}")

Model loaded: RandomForestClassifier
Test set: 1077 samples, 200 features
Label distribution: {np.False_: np.int64(669), np.True_: np.int64(408)}


/Users/melih.gorgulu/miniconda3/envs/aftercourt-automation/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/melih.gorgulu/miniconda3/envs/aftercourt-automation/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
# Predict probabilities
y_probs = model.predict_proba(X_test)[:, 1]
print(f"Prob stats — min: {y_probs.min():.4f}, max: {y_probs.max():.4f}, mean: {y_probs.mean():.4f}")

Prob stats — min: 0.0000, max: 1.0000, mean: 0.3804


## Precision / Recall at Different Thresholds

In [3]:
# Compute precision, recall, f1 for a range of thresholds
thresholds = np.arange(0.05, 1.00, 0.05).round(2).tolist()
# Add fine-grained thresholds in the high range
thresholds = sorted(set(thresholds + [0.85, 0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99]))

records = []
for th in thresholds:
    y_pred = (y_probs >= th).astype(int)
    n_pred_pos = y_pred.sum()
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    records.append({
        'threshold': th,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'predicted_positive': int(n_pred_pos),
        'total': len(y_test),
    })

results = pd.DataFrame(records)
print(results.to_string(index=False, float_format='%.4f'))

 threshold  precision  recall     f1  predicted_positive  total
    0.0500     0.9379  1.0000 0.9680                 435   1077
    0.1000     0.9645  1.0000 0.9819                 423   1077
    0.1500     0.9737  1.0000 0.9867                 419   1077
    0.2000     0.9761  1.0000 0.9879                 418   1077
    0.2500     0.9784  1.0000 0.9891                 417   1077
    0.3000     0.9831  1.0000 0.9915                 415   1077
    0.3500     0.9831  1.0000 0.9915                 415   1077
    0.4000     0.9831  1.0000 0.9915                 415   1077
    0.4500     0.9879  1.0000 0.9939                 413   1077
    0.5000     0.9879  0.9975 0.9927                 412   1077
    0.5500     0.9927  0.9951 0.9939                 409   1077
    0.6000     0.9927  0.9951 0.9939                 409   1077
    0.6500     0.9926  0.9926 0.9926                 408   1077
    0.7000     0.9951  0.9902 0.9926                 406   1077
    0.7500     0.9951  0.9877 0.9914    

In [4]:
# Plot Precision, Recall and F1 vs Threshold
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results['threshold'], y=results['precision'],
    name='Precision', mode='lines+markers',
    line=dict(color='#636EFA', width=2.5), marker=dict(size=5),
))
fig.add_trace(go.Scatter(
    x=results['threshold'], y=results['recall'],
    name='Recall', mode='lines+markers',
    line=dict(color='#EF553B', width=2.5), marker=dict(size=5),
))
fig.add_trace(go.Scatter(
    x=results['threshold'], y=results['f1'],
    name='F1 Score', mode='lines+markers',
    line=dict(color='#00CC96', width=2.5), marker=dict(size=5),
))

# Highlight current production threshold (0.95)
fig.add_vline(x=0.95, line_dash='dash', line_color='grey', line_width=1.5,
              annotation_text='Prod TH=0.95', annotation_position='top left',
              annotation_font=dict(size=11, color='grey'))

fig.update_layout(
    title=dict(text='Ladung: Precision / Recall / F1 vs Probability Threshold', font=dict(size=18)),
    xaxis_title='Threshold',
    yaxis_title='Score',
    yaxis_range=[0, 1.05],
    xaxis=dict(dtick=0.05),
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white',
    height=550,
)
fig.show()

## Precision-Recall Curve (sklearn)

In [5]:
# sklearn precision_recall_curve (all thresholds)
prec_curve, rec_curve, th_curve = precision_recall_curve(y_test, y_probs)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rec_curve, y=prec_curve,
    mode='lines',
    line=dict(color='steelblue', width=2.5),
    name='PR Curve',
    hovertemplate='Recall: %{x:.3f}<br>Precision: %{y:.3f}<extra></extra>',
))

fig.update_layout(
    title=dict(text='Ladung: Precision-Recall Curve', font=dict(size=18)),
    xaxis_title='Recall',
    yaxis_title='Precision',
    xaxis_range=[0, 1.02],
    yaxis_range=[0, 1.05],
    template='plotly_white',
    height=500,
    showlegend=False,
)
fig.show()

## Key Thresholds Summary

In [10]:
# Summary table for key thresholds
key_ths = [0.5, 0.65, 0.7, 0.8, 0.85, 0.9, 0.92, 0.95, 0.98]
summary = results[results['threshold'].isin(key_ths)].copy()
summary['precision_pct'] = (summary['precision'] * 100).round(2)
summary['recall_pct'] = (summary['recall'] * 100).round(2)
summary['f1_pct'] = (summary['f1'] * 100).round(2)

# Print as Markdown table (copy-paste into Notion)
header = "| Threshold | Precision % | Recall % | F1 % | Pred Positive |"
separator = "|-----------|-------------|----------|------|---------------|"
rows = []
for _, r in summary.iterrows():
    rows.append(f"| {r['threshold']} | {r['precision_pct']} | {r['recall_pct']} | {r['f1_pct']} | {r['predicted_positive']} |")

md_table = "\n".join([header, separator] + rows)
print(md_table)

| Threshold | Precision % | Recall % | F1 % | Pred Positive |
|-----------|-------------|----------|------|---------------|
| 0.5 | 98.79 | 99.75 | 99.27 | 412.0 |
| 0.65 | 99.26 | 99.26 | 99.26 | 408.0 |
| 0.7 | 99.51 | 99.02 | 99.26 | 406.0 |
| 0.8 | 99.5 | 98.04 | 98.77 | 402.0 |
| 0.85 | 99.75 | 97.79 | 98.76 | 400.0 |
| 0.9 | 99.75 | 96.57 | 98.13 | 395.0 |
| 0.92 | 99.74 | 95.1 | 97.37 | 389.0 |
| 0.95 | 100.0 | 93.87 | 96.84 | 383.0 |
| 0.98 | 100.0 | 89.22 | 94.3 | 364.0 |


In [9]:
import plotly.graph_objects as go

# Split probs by true label
probs_pos = y_probs[y_test == 1]
probs_neg = y_probs[y_test == 0]

# --- Graph 1: Overall predicted probability distribution ---
fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=y_probs, nbinsx=50, marker_color='steelblue', opacity=0.8, name='All',
))
fig1.add_vline(x=0.95, line_dash='dash', line_color='grey', line_width=1.5,
               annotation_text='TH=0.95', annotation_position='top right',
               annotation_font=dict(size=10, color='grey'))
fig1.update_layout(
    title=dict(text='Ladung Model — Predicted Probability Distribution (All Samples)', font=dict(size=17)),
    xaxis_title='Predicted Probability',
    yaxis_title='Count',
    template='plotly_white',
    height=450,
    showlegend=False,
)
fig1.show()

# --- Graph 2: By true label ---
fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=probs_neg, nbinsx=50, marker_color='#EF553B', opacity=0.6, name='Not Ladung (y=0)',
))
fig2.add_trace(go.Histogram(
    x=probs_pos, nbinsx=50, marker_color='#636EFA', opacity=0.6, name='Ladung (y=1)',
))
fig2.add_vline(x=0.95, line_dash='dash', line_color='grey', line_width=1.5,
               annotation_text='TH=0.95', annotation_position='top right',
               annotation_font=dict(size=10, color='grey'))
fig2.update_layout(
    title=dict(text='Ladung Model — Predicted Probability Distribution by True Label', font=dict(size=17)),
    xaxis_title='Predicted Probability',
    yaxis_title='Count',
    barmode='overlay',
    template='plotly_white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig2.show()